# 3장 3강: Gradient Descent와 Learning Rate — 실습문제

## 실습 목표

- 주어진 기울기 공식에 숫자를 넣어 가중치와 편향을 바꿉니다.
- 학습률에 따라 이동하는 양과 손실이 어떻게 달라지는지 읽습니다.
- 전체 데이터와 작은 묶음을 사용할 때의 업데이트를 구별합니다.
- 반복 계산에서 수렴의 뜻을 이해하고, 검증 손실 표에서 학습 종료 시점을 찾습니다.

## 실습 환경 / 데이터

Python과 NumPy를 사용합니다. 외부 데이터 파일은 필요하지 않습니다. 제공된 「3장 3강: Gradient Descent와 Learning Rate」의 본문과 연습문제에 나온 식·숫자만 사용합니다.

## 실습 구성과 교안 대응

| 구분 | 실습 내용 | 교안 근거 |
|---|---|---|
| 필수 1 / 문제 1-1 | w와 b를 두 번 업데이트하기 | 1절 본문: x=2, y=10, w=3, b=1, 학습률 0.05 |
| 필수 1 / 문제 1-2 | 네 학습률의 3회 업데이트 결과 읽기 | 2절 본문: L=(10−2w)², 초기 w=3의 학습률 비교 |
| 필수 2 / 문제 2-1 | Batch와 Mini-Batch의 기본 계산 | 3절 본문 및 연습문제 3: X=[1,2,3,4], y=[2,4,6,8] |
| 필수 2 / 문제 2-2 | 제공 코드로 수렴 관찰하기 | 3절 본문: L=(w−5)², 초기 w=1, 학습률 0.1, 30회 업데이트 |
| 과제 / 문제 3-1 | 같은 계산 순서로 두 번 업데이트하기 | 연습문제 1: x=3, y=12, w=2, b=1, 학습률 0.02 |
| 과제 / 문제 3-2 | 검증 손실 표에서 최적 시점과 종료 시점 찾기 | 연습문제 3의 문제 6: 6개 Epoch의 손실 표, patience=3 |

**교안의 연습문제 전체를 그대로 옮긴 자료는 아닙니다.** 본문 예제와 연습문제 일부를 기초 수준의 필수 실습 4문제와 복습 과제 2문제로 재구성했습니다. 사용한 개념과 계산 방법은 해당 강의의 범위 안에 있습니다.

## 진행 안내

1. 실습 준비 셀을 먼저 실행합니다.
2. 문제에 제공된 공식을 보고 숫자를 대입합니다. 미분 공식을 혼자 유도할 필요는 없습니다.
3. 문제 1-2와 2-2의 코드는 제공되어 있습니다. 실행한 뒤 표와 출력을 읽어 답하면 됩니다.
4. 다른 문제도 손으로 계산한 값을 코드로 확인하면 됩니다. 긴 프로그램을 직접 설계할 필요는 없습니다.
5. 각 문제의 Q1~Q4에 답합니다. 과제는 앞에서 배운 계산과 표 읽기만 복습합니다.
6. 소수는 계산 도중 반올림하지 않습니다. 출력에 긴 소수가 보이면 마지막에만 반올림합니다.

## 먼저 읽는 용어와 기호

| 용어·기호 | 쉬운 뜻 |
|---|---|
| 모델 / 예측 | 입력으로 정답을 추측하는 계산 규칙 / 그 규칙으로 계산한 값 |
| x / y | 입력 / 실제 정답. 여기서는 문제에서 주어지는 고정값 |
| X | 여러 입력을 순서대로 묶은 것 |
| ŷ / y_pred | 모델이 계산한 예측값. ŷ는 ‘와이 햇’으로 읽음 |
| w / b | 입력에 곱하는 가중치 / 예측에 더하는 편향 |
| 파라미터 | 학습하면서 조절하는 숫자. 이 실습에서는 w 또는 w와 b |
| L / loss | 예측과 정답의 차이를 평가하는 손실. 이 자료의 손실은 작을수록 오차가 작음 |
| MSE | 각 데이터의 ‘정답−예측’을 제곱한 뒤 평균 낸 손실 |
| 기울기 / grad | 현재 파라미터를 조금 바꿀 때 손실이 변하는 방향과 정도 |
| Gradient / ∇L | 여러 파라미터의 손실 기울기를 순서대로 묶은 것 |
| α / learning_rate | 학습률. ‘알파’라고 읽으며, 기울기에 곱해 업데이트 크기를 조절하는 양수 |
| 업데이트 | 현재 파라미터에서 ‘학습률×기울기’를 빼 새 값으로 바꾸는 일 |
| old / new | 업데이트 전 / 업데이트 후 |
| Batch / Mini-Batch / SGD | 전체 데이터 / 작은 데이터 묶음 / 데이터 하나로 기울기를 구해 업데이트하는 방식 |
| Epoch | 전체 학습 데이터를 한 번 사용한 단위. ‘에포크’라고 읽음 |
| Iteration | 파라미터를 한 번 업데이트하는 단위. ‘이터레이션’이라고 읽음 |
| 수렴 / Convergence | 반복하면서 파라미터나 손실이 일정한 값에 가까워지는 현상 |
| 학습 손실 / 검증 손실 | 학습에 사용하는 데이터의 손실 / 별도로 둔 검증 데이터의 손실 |
| Early Stopping / patience | 검증 손실이 좋아지지 않으면 학습을 일찍 멈추는 방법 / 연속으로 기다릴 횟수 |

### 수식을 읽는 방법

- `ŷ=wx+b`는 **w와 x를 곱한 뒤 b를 더한다**는 뜻입니다. 붙여 쓴 wx는 곱셈입니다.
- `L=(y−ŷ)²`는 **정답에서 예측을 뺀 값을 자기 자신과 곱한다**는 뜻입니다. 예를 들어 (10−7)²=3×3=9입니다.
- `∂L/∂w`는 w에 대한 손실의 기울기입니다. 기호가 분수처럼 생겼다고 해서 L을 w로 나누라는 뜻은 아닙니다. 코드에서는 `grad_w`라고 씁니다.
- `w_new=w_old−α×grad_w`는 **현재 w에서 학습률과 기울기의 곱을 뺀다**는 뜻입니다.
- `3−0.05×(−12)=3−(−0.6)=3+0.6=3.6`입니다. 음수를 빼면 더하기가 됩니다.
- `≈`는 ‘대략 같다’는 뜻입니다. 표의 긴 소수는 보기 좋게 반올림한 값일 수 있습니다.

학습률이 0.05라고 해서 언제나 0.05만큼 이동하지는 않습니다. 실제 변화량은 **−학습률×현재 기울기**입니다. 기울기를 계산하는 과정과 파라미터를 실제로 바꾸는 과정도 구별하세요.

## 실습 준비

아래 셀을 먼저 실행하세요. NumPy는 여러 숫자를 묶어서 계산하도록 돕는 도구입니다. `np`는 NumPy를 짧게 부르는 이름입니다.

In [ ]:
import numpy as np

---

## 기초 복습 과제

### 문제 3-1. 두 번의 업데이트 복습하기

### 문제 설명

교안 연습문제 1의 기본 계산을 복습합니다. 모델은 ŷ=wx+b, x=3, y=12입니다. 처음 w=2, b=1이고 학습률은 0.02입니다.

### 제공 공식

- 예측 ŷ=wx+b, 손실 L=(y−ŷ)²
- g_w=2(ŷ−y)x, g_b=2(ŷ−y)
- 새 w=w−0.02×g_w, 새 b=b−0.02×g_b

### 요구사항

- 초기 예측·손실·기울기를 계산합니다.
- 정확히 두 번 업데이트하고 매번 새 예측과 손실을 확인합니다.
- 문제 1-1의 순서와 코드를 참고해 숫자를 바꿔 계산하면 됩니다.
- 큰 학습률 비교나 추가 모델 구현은 요구하지 않습니다.

#### 해석 질문

**Q1.** 초기 예측, 손실, [w,b] 순서의 기울기는 얼마인가요?

**Q2.** 첫 업데이트 후 w, b, 예측, 손실은 얼마인가요?

**Q3.** 두 번째 기울기와 업데이트 후 w, b, 예측, 손실은 얼마인가요?

**Q4.** 왜 두 번째 업데이트에 첫 기울기를 그대로 사용하면 안 되나요?

#### 제출 결과

- 요구사항의 계산 결과 또는 제공 코드의 실행 결과
- Q1~Q4 답변

In [ ]:
# 문제 3-1. 두 번의 업데이트 복습하기
# 1. 문제의 초기값을 설정하세요.
# 2. 제공 공식에 숫자를 대입하세요.
# 3. 새 값과 결과를 출력하세요.

In [ ]:
w= 2
b= 1
x= 3
y= 12
y_hat= w * x + b
L= (y - y_hat)**2
g_w= 2 * (y_hat - y)*x
g_b= 2 * (y_hat - y)
w_new= w - 0.02 * g_w
b_new= b - 0.02 * g_b
print("y_hat: ", y_hat)
print("L: ", L)
print("g_w: ", g_w)
print("g_b: ", g_b)
print("w_new: ", w_new)
print("b_new: ", b_new)

y_hat:  7
L:  25
g_w:  -30
g_b:  -10
w_new:  2.6
b_new:  1.2


In [ ]:
w= w_new
b= b_new
x= 3
y= 12
y_hat= w * x + b
L= (y - y_hat)**2
g_w= 2 * (y_hat - y)*x
g_b= 2 * (y_hat - y)
w_new= w - 0.02 * g_w
b_new= b - 0.02 * g_b
print("y_hat: ", y_hat)
print("L: ", L)
print("g_w: ", g_w)
print("g_b: ", g_b)
print("w_new: ", w_new)
print("b_new: ", b_new)



y_hat:  9.0
L:  9.0
g_w:  -18.0
g_b:  -6.0
w_new:  2.96
b_new:  1.3199999999999998


#### 답변 작성란

**Q1.** 초기 예측, 손실, [w,b] 순서의 기울기는 얼마인가요?

답변: 
```python
g_w:  -30
g_b:  -10
```
[-30, -10]

**Q2.** 첫 업데이트 후 w, b, 예측, 손실은 얼마인가요?

답변: 
```python
w_new:  2.6
b_new:  1.2
y_hat:  9.0
L:  9.0
```

**Q3.** 두 번째 기울기와 업데이트 후 w, b, 예측, 손실은 얼마인가요?

답변: 
```python
g_w:  -18.0
g_b:  -6.0
w_new:  2.96
b_new:  1.3199999999999998
```

**Q4.** 왜 두 번째 업데이트에 첫 기울기를 그대로 사용하면 안 되나요?

답변: 첫 업데이트로 w, b가 바뀌면 예측이 7에서 9로 달라지고, 기울기도 (−30, −10)에서 (−18, −6)으로 바뀝니다. 첫 기울기를 그대로 쓰면 바뀐 상태가 반영되지 않아 w=3.2, b=1.4처럼 올바른 값(2.96, 1.32)과 다른 곳으로 이동합니다. 그래서 업데이트할 때마다 현재 상태에서 기울기를 다시 구해야 합니다. 

### 문제 3-2. 검증 손실 표로 학습 종료 시점 찾기

### 문제 설명

교안 연습문제 3의 손실 표를 읽습니다. 학습 손실은 학습 데이터의 손실이고, 검증 손실은 따로 둔 검증 데이터의 손실입니다. 이 문제에서는 검증 손실이 작을수록 좋은 상태로 판단합니다.

| Epoch | 학습 손실 | 검증 손실 |
|---|---:|---:|
| 1 | 0.80 | 0.60 |
| 2 | 0.65 | 0.45 |
| 3 | 0.50 | 0.40 |
| 4 | 0.40 | 0.42 |
| 5 | 0.32 | 0.43 |
| 6 | 0.25 | 0.44 |

### 사용할 규칙

- 지금까지 가장 작은 검증 손실보다 **더 작아졌을 때** 개선으로 봅니다.
- 개선하면 ‘연속 미개선 횟수’를 0으로 되돌립니다.
- 개선하지 않으면 그 횟수를 1 더합니다.
- patience=3: 연속으로 3개 Epoch에서 개선되지 않으면 그 Epoch가 끝난 뒤 학습을 멈춥니다.
- Epoch 1은 첫 기준값으로 기록하며, 그때의 연속 미개선 횟수는 0입니다.

### 요구사항

- 검증 손실이 가장 작은 Epoch과 그 값을 찾습니다.
- Epoch 4, 5, 6의 연속 미개선 횟수를 셉니다.
- 학습을 멈추는 Epoch과 검증 손실이 가장 좋았던 Epoch을 구별합니다.
- 표를 읽어 답하면 됩니다. 실제 모델 학습이나 Early Stopping 프로그램 작성은 필수가 아닙니다.

#### 해석 질문

**Q1.** 검증 손실이 가장 작은 Epoch과 그 손실은 얼마인가요?

**Q2.** Epoch 4, 5, 6의 연속 미개선 횟수는 각각 얼마인가요?

**Q3.** patience=3이면 언제 학습을 멈추며, 가장 좋은 모델의 시점은 언제인가요?

**Q4.** 학습 손실이 계속 줄어도 이 규칙에서는 왜 학습을 멈추나요?

#### 제출 결과

- 요구사항의 계산 결과 또는 제공 코드의 실행 결과
- Q1~Q4 답변

In [ ]:
# 문제 3-2. 검증 손실 표로 학습 종료 시점 찾기
# 제공 공식에 숫자를 대입하고 결과를 확인하세요.
# 문제 3-2는 표를 읽어 아래 답변란에 작성하면 됩니다.

In [ ]:
# 문제 3-2. 검증 손실 표로 학습 종료 시점 찾기
val_loss = [0.60, 0.45, 0.40, 0.42, 0.43, 0.44]  
patience = 3

best_loss = val_loss[0]   # Epoch 1은 첫 기준값으로 기록
best_epoch = 1
wait = 0                  # 연속 미개선 횟수
print("Epoch 1 / 검증 손실:", best_loss, "/ 기준값 기록 / 연속 미개선:", wait)

for epoch in range(2, len(val_loss) + 1):
    loss = val_loss[epoch - 1]
    if loss < best_loss:          # 지금까지 가장 작은 값보다 더 작아졌을 때만 개선
        best_loss = loss
        best_epoch = epoch
        wait = 0
        status = "개선"
    else:
        wait += 1
        status = "미개선"
    print("Epoch", epoch, "/ 검증 손실:", loss, "/", status, "/ 연속 미개선:", wait)
    if wait == patience:
        print("→ Epoch", epoch, "종료 후 학습 중단")
        break

print("가장 좋은 Epoch:", best_epoch, "/ 그때의 검증 손실:", best_loss)

Epoch 1 / 검증 손실: 0.6 / 기준값 기록 / 연속 미개선: 0
Epoch 2 / 검증 손실: 0.45 / 개선 / 연속 미개선: 0
Epoch 3 / 검증 손실: 0.4 / 개선 / 연속 미개선: 0
Epoch 4 / 검증 손실: 0.42 / 미개선 / 연속 미개선: 1
Epoch 5 / 검증 손실: 0.43 / 미개선 / 연속 미개선: 2
Epoch 6 / 검증 손실: 0.44 / 미개선 / 연속 미개선: 3
→ Epoch 6 종료 후 학습 중단
가장 좋은 Epoch: 3 / 그때의 검증 손실: 0.4


#### 답변 작성란

**Q1.** 검증 손실이 가장 작은 Epoch과 그 손실은 얼마인가요?

답변: **Epoch 3**, 검증 손실 **0.40**입니다. 

**Q2.** Epoch 4, 5, 6의 연속 미개선 횟수는 각각 얼마인가요?

답변: 순서대로 1, 2, 3입니다.

**Q3.** patience=3이면 언제 학습을 멈추며, 가장 좋은 모델의 시점은 언제인가요?

답변: **Epoch 6이 끝난 뒤** 학습을 멈춥니다. 가장 좋은 모델의 시점은 검증 손실이 가장 작았던 **Epoch 3**입니다. 종료 시점과 가장 좋은 시점은 다릅니다.

**Q4.** 학습 손실이 계속 줄어도 이 규칙에서는 왜 학습을 멈추나요?

답변: 이 규칙은 학습 손실이 아니라 **검증 손실**을 기준으로 삼기 때문입니다. 학습 손실은 계속 줄지만, 검증 손실은 Epoch 3 이후 계속 커집니다. 모델이 학습 데이터에 과적합되고 있다는 신호이므로, 더 학습해도 의미가 없다고 보고 멈춥니다.

---

## 실습 마무리

다음 질문으로 핵심 내용을 복습하세요.

**1.** 경사하강법의 기본 업데이트 식은 무엇인가요?

**2.** 학습률이 양수일 때 음수 기울기를 빼면 파라미터는 어떻게 되나요?

**3.** 학습률과 실제 변화량은 같은 값인가요?

**4.** Epoch와 Iteration은 어떻게 다른가요?

**5.** 수렴은 손실이 정확히 0이라는 뜻인가요?

**6.** Early Stopping의 종료 시점과 가장 좋은 검증 시점은 항상 같나요?

### 실습 마무리 답변 작성란

**1.** 답변: `w_new= w - learning_rate * (gradient)`

**2.** 답변: 양의 방향으로 증가한다.

**3.** 답변: 아닙니다. 무슨 변화량을 물어보는지 모르겠지만, 학습률은 계산된 gradient를 기준으로 얼만큼 파라미터를 이동시킬지 정하는 값입니다.

**4.** 답변: Epoch가 n이라는 건 전체 데이터를 n번 다 처리했다는 것을 의미한다. **iteration** Batch 하나로 파라미터를 한 번 업데이트하는 단위입니다.

**5.** 답변: 아닙니다. 수렴은 반복하면서 파라미터나 손실이 **일정한 값에 가까워지는 것**입니다. 손실이 0이 아닌 값에 수렴할 수도 있습니다.

**6.** 답변: **항상 같지 않습니다.** Early Stopping은 검증 손실이 개선되지 않은 상태로 patience만큼 연속해서 기다린 **뒤에** 멈추므로, 종료 시점은 보통 가장 좋은 검증 시점보다 patience만큼 뒤입니다. 문제 3-2에서도 가장 좋은 시점은 Epoch 3, 종료 시점은 Epoch 6(patience=3)이었습니다. 그래서 실제로는 가장 좋았던 시점의 모델을 기억해 두었다가 사용합니다.